# Assignment 06: A First-Principles Species Distribution Model

## BIO597 Spatial Analysis of Biodiversity

This assignment asks you to build and interpret a simple one-variable species distribution model without using an SDM package. You will choose one amphibian species, describe the environmental values at its occurrence records, compare those values with background environments, create a suitability curve, and project that curve across Maine.

## Deliverable

Submit a completed notebook with code, maps, plots, and short written interpretations. Your goal is to show that you understand the moving parts of an SDM before we use more automated modeling tools.

## Learning goals

By the end of this assignment, you should be able to:

* describe the difference between occurrence, background, and absence data
* extract one raster predictor at species occurrence records
* compare used environments with available environments
* create a simple environmental suitability score
* project a suitability score across raster space
* explain why a conceptual SDM is not the same as a validated prediction model

## Data

Use:

* `Amphibians_ME/Amphibians_ME.csv` for occurrence records
* `/home/jovyan/shared-readwrite/bioclim/` for one environmental variable of your choice that is NOT mean annual temperature
* `../labs/ME_ShapeFiles/Maine_State_Boundary.shp` for the Maine boundary

You may use the same species as the lab or choose a different species with at least 100 usable records.

## 1. Import packages

Import `pandas`, `geopandas`, `rioxarray`, `xarray`, `numpy`, `matplotlib.pyplot`, and `Path`.

In [4]:
%pip install geopandas rioxarray xarray matplotlib pandas numpy

  Using cached geopandas-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached rioxarray-0.23.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached xarray-2026.9.0-py3-none-any.whl.metadata (12 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
  Using cached pyproj-3.8.0-cp314-cp314-win_amd64.whl.metadata (40 kB)
  Using cached shapely-2.2.0-cp314-cp314-win_amd64.whl.metadata (6.8 kB)
  Using cached rasterio-1.5.2-cp314-cp314-win_amd64.whl.metadata (8.9 kB)
  Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.67.0-cp314-cp314-win_amd64.whl.metadata (139 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
# Import packages here.
from pathlib import Path

import geopandas as gpd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import numpy as np # type: ignore
import pandas as pd # type: ignore
import rioxarray as rxr # type: ignore
import xarray as xr # type: ignore

## 2. Load and combine the Maine boundary

Read the Maine state boundary shapefile, inspect its CRS and shape, and dissolve it to one row.

In [6]:
maine_boundary = gpd.read_file("C:/Users/Braec/Desktop/BIO597-SpatialBiodiversity/docs/labs/ME_ShapeFiles/Maine_State_Boundary.shp")
print(maine_boundary.shape)
print(maine_boundary.crs)

maine_boundary = maine_boundary.dissolve()
#maine_boundary.plot()

(6204, 10)
EPSG:4326


## 3. Open and clip environmental data

Select one environmental variable of your choosing that is NOT mean annual temperature. You can choose one you think will work well or one you think is less important for the study organism. Open the variable .tif file with `masked=True`, clip it to Maine, and remove the band dimension with `squeeze()`.


In [7]:
bioclim_path = Path("C:/Users/Braec/Desktop/BIO597-SpatialBiodiversity/bioclim")
#bioclim_path = Path("./bioclim")
bio1 = rxr.open_rasterio(
    bioclim_path / "wc2.1_2.5m_bio_1.tif",
    masked=True,
)

## 4. Map your environmental variable

Create a map of the clipped temperature raster. Include a title.

In [8]:
# Plot annual mean temperature for Maine.

bio1 = rxr.open_rasterio(bioclim_path / "wc2.1_2.5m_bio_1.tif", masked=True)

maine = maine_boundary.to_crs(bio1.rio.crs)
bio1_maine = (
    bio1.rio.clip_box(*maine.total_bounds)   # small subset first
        .rio.clip(maine.geometry, maine.crs, drop=True)
        .squeeze()
)
bio1_maine.name = "annual_mean_temperature_c"

## 5. Load and prepare amphibian occurrence records

Read the tab-delimited amphibian file. Keep `gbifID`, `species`, longitude, latitude, coordinate uncertainty, and year. Convert coordinates to numeric values and remove records missing species or coordinates.

In [10]:
amphibians = pd.read_csv(
    "../assignments/Amphibians_ME/Amphibians_ME.csv",
    sep="\t",
    low_memory=False,
)

columns_to_keep = [
    "gbifID",
    "species",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "year",
]

amphibians = amphibians[columns_to_keep].copy()
amphibians["decimalLongitude"] = pd.to_numeric(amphibians["decimalLongitude"], errors="coerce")
amphibians["decimalLatitude"] = pd.to_numeric(amphibians["decimalLatitude"], errors="coerce")

# Simple cleaning
amphibians = amphibians.dropna(subset=["species", "decimalLongitude", "decimalLatitude"]).copy()

# Reduce spatial imprecision (The "tulipifera-ike" filter!)
amphibians = amphibians[amphibians["coordinateUncertaintyInMeters"] < 200]

amphibians["species"].value_counts().head(12)

species
Pseudacris crucifer          1839
Aquarana clamitans           1779
Anaxyrus americanus          1689
Boreorana sylvatica          1617
Lithobates catesbeianus      1487
Ambystoma maculatum          1317
Plethodon cinereus           1283
Lithobates palustris         1049
Dryophytes versicolor         790
Notophthalmus viridescens     611
Lithobates pipiens            326
Hemidactylium scutatum        233
Name: count, dtype: int64

## 6. Choose one species

Display the most common species, then choose one species with at least 100 records. Store its scientific name as `target_species`.

In [ ]:
# Display species counts.

In [ ]:
target_species = "____________________________"

# Filter the occurrence table to your target species.
# Display the number of records and the first five rows.

**Why did you choose this species?** Include one sentence about record count and one sentence about why it may or may not be appropriate for this simple model.

## 7. Create an occurrence GeoDataFrame

Use longitude and latitude to create point geometry. Assign `EPSG:4326`. Confirm that the occurrence CRS and raster CRS are compatible.

In [ ]:
# Create presence_gdf.
# Print the occurrence CRS and raster CRS.

**CRS check:** Explain why coordinate systems matter before extracting raster values at points.

## 8. Extract environment at occurrence points

Create `xarray.DataArray` objects for point longitude and latitude. Use `.sel(..., method="nearest")` to extract BIO1 values at occurrence records. Add the extracted values to a column named `bio1_c`.

In [ ]:
# Create point_x and point_y.
# Extract temperature values.
# Add them to presence_gdf as bio1_c.
# Drop records with missing bio1_c.
# Rebuild point_x and point_y after dropping missing values.

## 9. Summarize occurrence environment

Calculate the count, mean, standard deviation, minimum, and maximum of `bio1_c` at occurrence records.

In [ ]:
# Summarize occurrence temperature values.

**Interpretation:** In two or three sentences, describe the environmental range represented by your species records.

## 10. Sample background environment

Create a random sample of up to 5,000 valid cells from Maine for your environmental variable. Use a random seed so your results are reproducible.

In [ ]:
rng = np.random.default_rng(597)

# Create an array of valid raster environmental values.
# Randomly sample up to 5000 cells without replacement.
# Print the background sample size, mean, and standard deviation.

## 11. Compare occurrence and background environments

Make overlapping histograms of occurrence environment and background environment. Use density-scaled histograms so the shapes can be compared even though the sample sizes differ.

In [ ]:
# Create an overlapping histogram of background and occurrence temperatures.

**Interpretation:** Does your species appear to use a narrow or broad subset of available environmental space? Explain using your plot.

## 12. Define a suitability function

Write a function that takes `observed`, `optimum`, and `tolerance` and returns:

```python
np.exp(-0.5 * ((observed - optimum) / tolerance) ** 2)
```

This creates a score near 1 when observed environment is close to the optimum and closer to 0 as temperature gets farther away.

In [ ]:
def environmental_suitability(observed, optimum, tolerance):
    # Replace this comment with the suitability calculation.
    return __________

## 13. Plot your suitability curve

Use the mean occurrence environmental value as the optimum and the standard deviation of occurrence as the tolerance. Plot suitability across the full range of Maine BIO1 values.

In [ ]:
# Calculate presence_mean and presence_sd.
# Create a sequence of environmental values across the raster range.
# Calculate suitability for those values.
# Plot the curve.

**Interpretation:** What value receives the highest suitability score? What does the tolerance control?

## 14. Project suitability across Maine

Apply your suitability function to every cell in the environmental raster. Keep missing raster cells missing.

In [ ]:
# Use xr.apply_ufunc() to apply environmental_suitability to the raster.
# Store the result as suitability.
# Mask cells where the environmental raster is missing.

## 15. Map suitability

Create a continuous suitability map with values from 0 to 1.

In [ ]:
# Plot the suitability raster.

## 16. Overlay occurrence points

Plot your suitability raster and overlay the occurrence points. This is a visual check, not a formal validation.

In [ ]:
# Plot suitability and overlay presence_gdf points.

**Visual check:** Do occurrence points generally fall in areas with higher predicted suitability? Describe any mismatch you see.

## 17. Compare occurrence and background suitability scores

Extract suitability at occurrence records. Also calculate suitability for your background environmental values. Compare the two distributions with histograms.

In [ ]:
# Extract suitability at occurrence points.
# Calculate suitability for background environmental values.
# Print mean occurrence suitability and mean background suitability.
# Plot overlapping histograms.

**Interpretation:** Did your model score occurrence records higher than background locations? What would it mean if it did not?

## 18. Change the tolerance

Create two additional suitability rasters: one with half the observed standard deviation and one with twice the observed standard deviation. Map all three side by side.

In [ ]:
# Create narrow_suitability using presence_sd / 2.
# Create broad_suitability using presence_sd * 2.
# Plot narrow, original, and broad suitability maps side by side.

**Interpretation:** How does tolerance affect the predicted distribution? Which tolerance seems most defensible for your species, and why?

## 19. Threshold the continuous map

Convert your continuous suitability map to a binary suitable/unsuitable map using a threshold of your choice. Report the fraction of Maine climate cells labeled suitable.

In [ ]:
threshold = __________

# Create a binary suitability map.
# Plot it.
# Calculate the fraction of non-missing cells labeled suitable.

**Threshold justification:** Why did you choose this threshold? How would the map change with a lower or higher threshold?

## 20. Save your outputs

Save your continuous suitability raster as `Assignment06_suitability.tif`. Save a CSV containing your occurrence records and extracted BIO1 values as `Assignment06_presence.csv`.

In [ ]:
# Save the suitability raster.
# Save the occurrence table with extracted temperature values.

## Final reflection

Answer each question in two or three sentences.

1. In this exercise, what information came from occurrence records and what information came from the environmental raster?
2. Why did we compare occurrence environment with background environment?
3. Why is this suitability score not the same as probability of presence?
4. Name two ecological or sampling processes ignored by this one-variable model.
5. What would a full SDM package add beyond this first-principles workflow?

## Submit your work

Run the notebook from top to bottom. Make sure all maps, plots, saved files, and written answers are complete. Then commit and push the completed notebook to your class GitHub repository.

```bash
git add docs/assignments/Assignment-06-FirstPrinciplesSDM.ipynb
git commit -m "Complete first principles SDM assignment"
git push
```